In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

sys.path.append(str(Path.cwd().parent))
from src.evaluation import *
from src.model import Autoencoder

In [2]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

In [3]:
svm = joblib.load("../models/one_class_svm.joblib")
is_forest = joblib.load("../models/isolation_forest.joblib")
autoencoder = Autoencoder(31, 8)
state_dict = torch.load("../models/autoencoder.pth")
autoencoder.load_state_dict(state_dict)

<All keys matched successfully>

In [6]:
cols_with_high_corr = ["V17", "V14", "V12", "V10"]

baseline_anomaly_score = X_test[cols_with_high_corr].abs().sum(axis=1)
baseline_score = average_precision_score(y_test, baseline_anomaly_score)

svm_score = evaluate(svm, X_test, y_test)
is_forest_score = evaluate(is_forest, X_test, y_test)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
autoencoder_score, _ = evaluate_autoencoder(autoencoder, X_test_tensor, y_test)

print(f"Baseline score: {baseline_score}")
print(f"SVM score: {svm_score}")
print(f"Isolation Forest score: {is_forest_score}")
print(f"Autoencoder score: {autoencoder_score}")

Baseline score: 0.7729532733460739
SVM score: 0.333778068492517
Isolation Forest score: 0.5828954032670963
Autoencoder score: 0.7704072160922986
